In [2]:
import os
import glob

import numpy as np
import pandas as pd

from tqdm import tqdm
from scipy.stats import skew, kurtosis
from scipy.signal import savgol_filter

In [3]:
import sys
sys.path.append("..") 
from pre_processing import  create_vitaldb_feature_set, extract_features_from_dataset

/data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
BASE_DIR = "/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/5sec_50%overlap/vitaldb_checkpoints"
UCI_PREPROCESSING_DIR = (
    "/data1/yashvi_bhuva/BP_estimation_using_PPG/"
    "UCI/Feature_extraction/Non_fiducial_features/"
    "5sec_50%overlap/data"
)
checkpoint_files = sorted(
    glob.glob(
        os.path.join(
            BASE_DIR,
            "case_*.npz"
        )
    )
)

print(
    "Number of VitalDB case files:",
    len(checkpoint_files)
)

print("\nFirst few files:")
for f in checkpoint_files[:5]:
    print(f)

Number of VitalDB case files: 1963

First few files:
/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/5sec_50%overlap/vitaldb_checkpoints/case_1.npz
/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/5sec_50%overlap/vitaldb_checkpoints/case_10.npz
/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/5sec_50%overlap/vitaldb_checkpoints/case_1001.npz
/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/5sec_50%overlap/vitaldb_checkpoints/case_1002.npz
/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/5sec_50%overlap/vitaldb_checkpoints/case_1003.npz


In [5]:
data = np.load(
    checkpoint_files[0]
)

print("Keys:")
print(data.files)

X = data["X"]
y = data["y"]

print("\nX shape:", X.shape)
print("y shape:", y.shape)

print("\nX dtype:", X.dtype)
print("y dtype:", y.dtype)

Keys:
['X', 'y', 'case_id']

X shape: (3963, 625)
y shape: (3963, 2)

X dtype: float32
y dtype: float32


In [6]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from joblib import Parallel, delayed


# --------------------------------------------------
# Find all cases
# --------------------------------------------------

npz_files = sorted(
    glob.glob(os.path.join(BASE_DIR, "case_*.npz"))
)

print("Number of .npz files:", len(npz_files))


# --------------------------------------------------
# Function to process ONE case
# --------------------------------------------------

def process_case(file):

    data = np.load(file)

    X = data["X"]
    y = data["y"]

    df_case = extract_features_from_dataset(
        X,
        y,
        fs=125,
        n_jobs=1
    )

    case_id = os.path.basename(file).replace(".npz", "")

    df_case["case_id"] = case_id

    return df_case


# --------------------------------------------------
# Parallel processing + tqdm
# --------------------------------------------------

parallel = Parallel(
    n_jobs=8,
    backend="loky",
    return_as="generator_unordered"
)

results = parallel(
    delayed(process_case)(file)
    for file in npz_files
)

# --------------------------------------------------
# Collect results while showing progress
# --------------------------------------------------

all_dfs = []

for df_case in tqdm(
    results,
    total=len(npz_files),
    desc="Extracting features",
    unit="case"
):
    all_dfs.append(df_case)


# --------------------------------------------------
# Combine all cases
# --------------------------------------------------

df_vitaldb = pd.concat(
    all_dfs,
    ignore_index=True
)

print("Final shape:", df_vitaldb.shape)
print(df_vitaldb.head())

Number of .npz files: 1963


Extracting features: 100%|██████████| 1963/1963 [3:07:24<00:00,  5.73s/case]  


Final shape: (9271840, 60)
   ppg_mean  ppg_median   ppg_std  ppg_variance   ppg_iqr  ppg_skewness  \
0  0.486242    0.429013  0.300184      0.090111  0.592751      0.259807   
1  0.469296    0.394508  0.298438      0.089065  0.571475      0.344543   
2  0.472995    0.401684  0.298145      0.088890  0.573282      0.344983   
3  0.478562    0.409509  0.303115      0.091879  0.588793      0.302922   
4  0.474208    0.405497  0.297868      0.088725  0.577169      0.335045   

   ppg_kurtosis  ppg_zero_crossing_rate  ppg_shannon_entropy  ppg_energy_mean  \
0     -1.467045                     0.0             5.422543         0.326542   
1     -1.386307                     0.0             5.399714         0.309304   
2     -1.370338                     0.0             5.434132         0.312615   
3     -1.431625                     0.0             5.367867         0.320900   
4     -1.398395                     0.0             5.420729         0.313599   

   ...  apg_energy_kurtosis  apg_en

In [7]:
VITALDB_SAVE_DIR = "/data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/5sec_50%overlap/data"
os.makedirs(VITALDB_SAVE_DIR, exist_ok=True)

In [8]:
SBP_FTEST_vitaldb = create_vitaldb_feature_set(df_vitaldb, UCI_PREPROCESSING_DIR, VITALDB_SAVE_DIR, "FTEST", "SBP")
SBP_MRMR_vitaldb = create_vitaldb_feature_set(df_vitaldb, UCI_PREPROCESSING_DIR, VITALDB_SAVE_DIR, "MRMR", "SBP")
SBP_RELIEFF_vitaldb = create_vitaldb_feature_set(df_vitaldb, UCI_PREPROCESSING_DIR, VITALDB_SAVE_DIR, "RELIEFF", "SBP")

DBP_FTEST_vitaldb = create_vitaldb_feature_set(df_vitaldb, UCI_PREPROCESSING_DIR, VITALDB_SAVE_DIR, "FTEST", "DBP")
DBP_MRMR_vitaldb = create_vitaldb_feature_set(df_vitaldb, UCI_PREPROCESSING_DIR, VITALDB_SAVE_DIR, "MRMR", "DBP")
DBP_RELIEFF_vitaldb = create_vitaldb_feature_set(df_vitaldb, UCI_PREPROCESSING_DIR, VITALDB_SAVE_DIR, "RELIEFF", "DBP")

Processing: SBP_FTEST
Loading: /data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/Feature_extraction/Non_fiducial_features/5sec_50%overlap/data/SBP_FTEST_preprocessing.pkl
Target: SBP
Number of features: 15
Features:
['vpg_iqr', 'vpg_skewness', 'vpg_energy_iqr', 'apg_zero_crossing_rate', 'apg_energy_iqr', 'ppg_iqr', 'vpg_shannon_entropy', 'apg_skewness', 'apg_shannon_entropy', 'vpg_kte_iqr', 'apg_iqr', 'vpg_energy_skewness', 'ppg_kte_iqr', 'vpg_kurtosis', 'ppg_energy_iqr']

VitalDB shape: (9271840, 17)
Saved: /data1/yashvi_bhuva/BP_estimation_using_PPG/VitalDB/ML/non_fiducial_features/5sec_50%overlap/data/SBP_FTEST_vitaldb.csv

Processing: SBP_MRMR
Loading: /data1/yashvi_bhuva/BP_estimation_using_PPG/UCI/Feature_extraction/Non_fiducial_features/5sec_50%overlap/data/SBP_MRMR_preprocessing.pkl
Target: SBP
Number of features: 15
Features:
['vpg_iqr', 'ppg_zero_crossing_rate', 'apg_energy_variance', 'ppg_energy_kurtosis', 'vpg_mean', 'ppg_std', 'vpg_median', 'apg_energy_kurtosis', 'apg_median